In [1]:
"""
AI Learning Navigator — V2
Personalized Learning Path Recommender.

Difference from V1: instead of returning just the next single step,
this returns the FULL ordered path from where the learner is now
to their career goal.

Logic:
1. Pick the "goal topic" — the most advanced topic in the graph that
   matches the learner's goal (e.g. for "robotics" -> robotics_control).
2. Walk backwards from the goal topic through prerequisites to collect
   every topic needed that isn't completed yet.
3. Topologically sort that set so every topic appears after its
   prerequisites (this gives a valid, learnable order).
"""

import json
from pathlib import Path


class LearningPathBuilder:
    def __init__(self, topics_path: str = "topics.json"):
        data = json.loads(Path(topics_path).read_text(encoding="utf-8"))
        self.topics = data["topics"]

    def _depth(self, topic_id: str, _cache: dict = {}) -> int:
        """Distance from the root of the graph (topics with no prerequisites)."""
        if topic_id in _cache:
            return _cache[topic_id]
        prereqs = self.topics[topic_id]["prerequisites"]
        depth = 0 if not prereqs else 1 + max(self._depth(p) for p in prereqs)
        _cache[topic_id] = depth
        return depth

    def _goal_topic(self, goal: str) -> str:
        """Pick the deepest topic relevant to this goal (the true 'end point' of the path,
        not just any topic tagged 'advanced' -- ties on level are broken by actual graph depth)."""
        matches = [tid for tid, info in self.topics.items() if goal in info["goals"]]
        if not matches:
            raise ValueError(f"No topics found for goal '{goal}'")
        return max(matches, key=self._depth)

    def _collect_needed(self, target: str, completed: set) -> set:
        """Walk backwards from target through prerequisites, collecting anything not done."""
        needed = set()
        stack = [target]
        while stack:
            current = stack.pop()
            if current in completed or current in needed:
                continue
            needed.add(current)
            stack.extend(self.topics[current]["prerequisites"])
        return needed

    def _topological_sort(self, needed: set) -> list[str]:
        """Order `needed` topics so every topic comes after its prerequisites."""
        visited = set()
        order = []

        def visit(topic_id):
            if topic_id in visited:
                return
            visited.add(topic_id)
            for prereq in self.topics[topic_id]["prerequisites"]:
                if prereq in needed:
                    visit(prereq)
            order.append(topic_id)

        for topic_id in needed:
            visit(topic_id)
        return order

    def build_path(self, completed: list[str], goal: str) -> dict:
        completed_set = set(completed)
        target = self._goal_topic(goal)

        if target in completed_set:
            return {"goal": goal, "target": target, "path": [], "message": "الهدف محقق بالفعل!"}

        needed = self._collect_needed(target, completed_set)
        ordered_ids = self._topological_sort(needed)

        path = []
        for i, topic_id in enumerate(ordered_ids, start=1):
            info = self.topics[topic_id]
            path.append({"step": i, "id": topic_id, "name": info["name"], "level": info["level"]})

        return {
            "goal": goal,
            "target": self.topics[target]["name"],
            "total_steps": len(path),
            "path": path,
        }


if __name__ == "__main__":
    builder = LearningPathBuilder("topics.json")

    result = builder.build_path(
        completed=["python", "math", "statistics", "data_analysis", "ml_basics"],
        goal="robotics",
    )

    print(f"Goal: {result['goal']} -> Target topic: {result['target']}")
    print(f"Total steps: {result['total_steps']}\n")
    print("Recommended Learning Path:")
    for step in result["path"]:
        print(f"  {step['step']}. {step['name']} ({step['level']})")

Goal: robotics -> Target topic: Robotics: Control Systems & ROS
Total steps: 6

Recommended Learning Path:
  1. First ML Project (Regression/Classification) (intermediate)
  2. Deep Learning Fundamentals (intermediate)
  3. PyTorch / TensorFlow (intermediate)
  4. Build a CNN Project (intermediate)
  5. Computer Vision (advanced)
  6. Robotics: Control Systems & ROS (advanced)
